# 07 — Production Tuning Experiments

Goal: reason about how vLLM settings change memory pressure and concurrency. Some settings require restarting the vLLM server. This notebook gives a disciplined experiment plan instead of random flag-tweaking.

Important knobs:

- `max_model_len`: maximum allowed context length. Larger values increase KV-cache pressure.
- `max_num_seqs`: maximum active sequences. Higher values can improve concurrency but require more KV cache.
- `max_num_batched_tokens`: token budget per scheduling step.
- `gpu_memory_utilization`: fraction of GPU memory vLLM may reserve for model/cache usage.
- quantisation: reduces weight memory, but not necessarily KV-cache memory.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

import pandas as pd
from vllm_lab.kv_cache import COMMON_MODEL_SHAPES, kv_cache_bytes_with_gqa, bytes_to_gib

Estimate KV-cache memory under different tuning choices. This does not replace real profiling, but it prevents reckless configurations.

In [ ]:
shape = COMMON_MODEL_SHAPES['tinyllama_1_1b']
rows = []
for max_model_len in [1024, 2048, 4096, 8192]:
    for max_num_seqs in [4, 8, 16, 32, 64]:
        b = kv_cache_bytes_with_gqa(
            num_layers=shape.num_layers,
            hidden_size=shape.hidden_size,
            num_heads=shape.num_heads,
            num_kv_heads=shape.num_kv_heads,
            context_tokens=max_model_len,
            bytes_per_value=2,
            active_sequences=max_num_seqs,
        )
        rows.append({'max_model_len': max_model_len, 'max_num_seqs': max_num_seqs, 'estimated_kv_cache_gib': bytes_to_gib(b)})

df = pd.DataFrame(rows)
df.pivot(index='max_model_len', columns='max_num_seqs', values='estimated_kv_cache_gib')

Experiment matrix. Restart the server for each row, run the stress test, then compare the generated CSVs in the dashboard.

In [ ]:
experiment_matrix = pd.DataFrame([
    {'name': 'low_memory_low_concurrency', 'max_model_len': 1024, 'max_num_seqs': 8, 'gpu_memory_utilization': 0.80},
    {'name': 'balanced_default', 'max_model_len': 2048, 'max_num_seqs': 16, 'gpu_memory_utilization': 0.85},
    {'name': 'higher_concurrency', 'max_model_len': 2048, 'max_num_seqs': 32, 'gpu_memory_utilization': 0.90},
    {'name': 'long_context_penalty', 'max_model_len': 8192, 'max_num_seqs': 8, 'gpu_memory_utilization': 0.90},
])
experiment_matrix

In [ ]:
for _, r in experiment_matrix.iterrows():
    cmd = (
        f"MODEL=TinyLlama/TinyLlama-1.1B-Chat-v1.0 "
        f"MAX_MODEL_LEN={int(r.max_model_len)} "
        f"MAX_NUM_SEQS={int(r.max_num_seqs)} "
        f"GPU_MEMORY_UTILIZATION={r.gpu_memory_utilization} "
        f"./scripts/run_vllm_server.sh"
    )
    print('---', r['name'])
    print(cmd)

Judgement rule: the best configuration is not the one with the largest throughput number. It is the smallest stable configuration that meets your latency, throughput, context-length, and concurrency requirements.